> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [モデル探索 (Discover)](#モデル探索-discover)
- [モデルの比較とデプロイ](#モデルの比較とデプロイ)
- [Embeddingモデルのデプロイ](#embeddingモデルのデプロイ)
- [Model Routerのデプロイ](#model-routerのデプロイ)
- [Model Routerの構成](#model-routerの構成)

## 🎯 学習目標

- モデルリーダーボードを通じたモデル性能の比較
- 様々なAIモデルのデプロイ方法の理解
- Model Routerの設定と構成
- モデルルーティング戦略の理解

## ⏱️ 予想所要時間

約15分

## 環境設定

まず、前のノートブックで作成したFoundryリソース情報を設定します。

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを 見つけられるように)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# が前 ノートブックで 保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (他のツールが使用できるように)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つかりません.")
    print("💡 01-setup.ipynbを 先に実行して環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import InteractiveBrowserCredential

print(f"\n💡 使用するプロジェクトエンドポイント: {PROJECT_ENDPOINT}")

## 利用可能なモデルの取得

Azure CLIで Sweden Central リージョンで 使用 可能な モデルを 確認します.

In [ ]:
# Sweden Central リージョンで 使用可能なモデルの取得
print("🔍 Sweden Central リージョンで 使用 可能な AI モデル 取得 中...")
print("=" * 80)

# Azure CLIで リージョンで 使用可能なモデルの取得
# GPT 関連 モデル フィルターリングして 表示 簡単に 表示
import subprocess

cmd = [
    "az", "cognitiveservices", "model", "list",
    "--location", LOCATION,
    "--query", "[?contains(model.name, 'gpt') || contains(model.name, 'embedding')].{Name:model.name, Version:model.version, Format:model.format, Kind:kind}",
    "--output", "table"
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.stderr and not result.stderr.startswith("("):  # Azure CLI warnings 無視
    print(result.stderr)

print("\n" + "=" * 80)
print("💡 全体 モデル リストを 見るには 以下 コマンドを 実行してください:")
print(f"   !az cognitiveservices model list --location {LOCATION} --output table")
print("\n💡 上 リストで 望むは モデルを 選択して 以下 セルで デプロイする できる あります.")
print("💡 モデル 名前(Name)と バージョン(Version)を 確認してください.")

## GPT-4.1モデルのデプロイ

GPT-4.1 モデルを デプロイします. GlobalStandard SKUを 使用して最適なパフォーマンスを 提供します.

In [ ]:
# GPT-4.1 モデル デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-4.1 \
    --model-name gpt-4.1 \
    --model-format OpenAI \
    --model-version "2025-04-14" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ GPT-4.1 モデル デプロイ 完了!")

## GPT-5.1モデルのデプロイ

GPT-5.1 モデルを デプロイします. GlobalStandard SKUを 使用して最適なパフォーマンスを 提供します.

In [ ]:
# GPT-5.1 モデル デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-5.1 \
    --model-name gpt-5.1 \
    --model-format OpenAI \
    --model-version "2025-11-13" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ GPT-5.1 モデル デプロイ 完了!")

## デプロイ状態の確認

デプロイされた モデルの 状態を 確認します.

In [ ]:
# デプロイ 状態 確認
!az cognitiveservices account deployment show \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-4.1 \
    --query "{Name:name, Model:properties.model.name, Version:properties.model.version, Status:properties.provisioningState}" \
    --output table

# すべての デプロイされた モデル リスト
print("\n📦 デプロイされた すべての モデル:")
!az cognitiveservices account deployment list \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --output table

## Embeddingモデルのデプロイ

Embedding モデルは テキストを ベクトルで 変換して の美的 検索 および 類似も 計算に 使用なります.

In [ ]:
# text-embedding-3-large モデル デプロイ (ベクトル 次元: 3072)
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name text-embedding-3-large \
    --model-name text-embedding-3-large \
    --model-format OpenAI \
    --model-version "1" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ Embedding モデル デプロイ 完了!")

# デプロイ 状態 確認
!az cognitiveservices account deployment show \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name text-embedding-3-large \
    --query "{Name:name, Model:properties.model.name, Status:properties.provisioningState}" \
    --output table
print("   ベクトル 次元: 3072 (Knowledge Baseで 使用)")

## Model Routerのデプロイ

Model Routerは 複数の モデル 間の インテリジェント ルーティングを 提供して コスト, 品質, パフォーマンスを 最適化します.

### Routing Mode オプション:

**a) Balanced Mode (バランス モード)** - デフォルト値
- コスト, 品質, パフォーマンスの バランス 維持
- 一般的な 本番環境 ワークロードに 適合

**b) Quality Mode (品質 モード)**
- 最高 品質の レスポンス まず
- 正確もが 重要した アプリケーション

**c) Cost Mode (コスト モード)**
- コスト 最適化 まず
- 大量の 簡単な リクエスト 処理

In [ ]:
# Model Router デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name model-router \
    --model-name model-router \
    --model-format OpenAI \
    --model-version "2025-11-18" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ Model Router デプロイ 完了!")
print("\n📊 Model Router 動作 方式:")
print("  ユーザー リクエスト → Model Router → 判断:")
print("    - 簡単な 質問 → 低コスト モデル")
print("    - 複雑な 分析 → 高品質 モデル")
print("    - 高いは 負荷 → 負荷 分散")

## デプロイされたモデルの最終確認

すべての モデルが 正常にで デプロイされているか 確認します.

In [ ]:
# 最終 デプロイ リスト 確認
print("=" * 80)
print("デプロイされた モデル リスト")
print("=" * 80)

!az cognitiveservices account deployment list \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --query "[].{Name:name, Model:properties.model.name, SKU:sku.name, Status:properties.provisioningState}" \
    --output table

print("\n✅ 次の モデルが デプロイされてよ します:")
print("  1. gpt-4.1 (Language Model)")
print("  2. gpt-5.1 (Language Model)")
print("  3. text-embedding-3-large (Embedding Model)")
print("  4. model-router (Router)")

print("\n💡 ポータル 確認: https://ai.azure.com")
print("   Build > Modelsで デプロイされた モデルを 視覚的で 確認する できる あります.")

## 次のステップ

モデル デプロイが 完了なりました! が第 が モデルたちを 活用して エージェントを 構築してみましょう:

➡️ **[03. エージェント 開発](./03-agents.ipynb)**: 様々な 機能を が進 AI エージェントを だけ聞いてみます.